In [0]:
from pyspark.sql import functions as F

staging_order_items = spark.table("brazilian_ecommerce.prod.staging_order_items")
staging_products = spark.table("brazilian_ecommerce.prod.staging_products")
staging_category_translation = spark.table("brazilian_ecommerce.prod.staging_category_translation")
staging_orders = spark.table("brazilian_ecommerce.prod.staging_orders")

gold_revenue_by_category_month = (
    staging_order_items
    .join(staging_orders.select("order_id", "order_purchase_timestamp"), on="order_id", how="left")
    .join(staging_products.select("product_id", "product_category_name"), on="product_id", how="left")
    .join(staging_category_translation, on="product_category_name", how="left")
    .withColumn(
        "product_category_name_english",
        F.coalesce(
            F.col("product_category_name_english"),
            F.col("product_category_name"),   # fallback: untranslated name if no match found
            F.lit("uncategorized"),            # final fallback: product has no category at all
        )
    )
    .withColumn("year_month", F.date_format("order_purchase_timestamp", "yyyy-MM"))
    .groupBy("year_month", "product_category_name_english")
    .agg(
        F.sum("price").alias("total_revenue"),
        F.countDistinct("order_id").alias("total_orders"),
        F.count("order_item_id").alias("total_units_sold"),
    )
    .withColumn("avg_ticket_value", (F.col("total_revenue") / F.col("total_orders")).cast("decimal(10,2)"))
    .orderBy("year_month", "product_category_name_english")
)

full_table_name = "brazilian_ecommerce.prod.gold_revenue_by_category_month"

(
    gold_revenue_by_category_month.write
    .mode("overwrite")
    .option("comment", "gold table: revenue, orders and units sold by product category and month")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

print(f"Loaded {gold_revenue_by_category_month.count()} rows into {full_table_name}")